In [1]:
from flask import Flask, request, jsonify, render_template_string
import os
import re
import json
import oracledb
import anthropic

app = Flask(__name__)

# -----------------------------
# Configuration
# -----------------------------
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_API_KEY")

DB_USER = os.getenv("ORACLE_DB_USER", "YOUR_DATABASE_USER")
DB_PASSWORD = os.getenv("ORACLE_DB_PASSWORD", "YOUR_DATABASE_PASSWORD")
DB_DSN = os.getenv("ORACLE_DB_DSN", "localhost:1521/FREEPDB1")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """
You are a biomedical data assistant.
You help users query an Oracle database containing cancer gene expression data.

Available tables:
- patient_labels(sample_id, cancer_type)
- gene_expression(sample_id, gene_id, expression_value)
- pca_results(sample_id, cancer_type, pca1, pca2)
- tsne_results(sample_id, cancer_type, tsne_x, tsne_y)
- cluster_results(sample_id, cancer_type, cluster_id)

Rules:
- Generate only read-only SQL.
- Never use INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, MERGE, GRANT, REVOKE.
- Use only the available tables listed above.
- Prefer concise Oracle SQL.
- If asked for SQL, return SQL only.
- When given query results, explain them clearly for a biomedical audience.
"""

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Claude Cancer Chat</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 40px;
            max-width: 900px;
        }
        textarea, input[type="text"] {
            width: 100%;
            padding: 10px;
            font-size: 16px;
        }
        button {
            padding: 10px 18px;
            font-size: 16px;
            margin-top: 10px;
            cursor: pointer;
        }
        .box {
            border: 1px solid #ccc;
            border-radius: 8px;
            padding: 16px;
            margin-top: 20px;
            background: #fafafa;
        }
        pre {
            white-space: pre-wrap;
            word-wrap: break-word;
            background: #f4f4f4;
            padding: 12px;
            border-radius: 6px;
            overflow-x: auto;
        }
        .error {
            color: #b00020;
            font-weight: bold;
        }
    </style>
</head>
<body>
    <h2>Claude Cancer Chat</h2>
    <p>Ask a question about the cancer gene expression dataset.</p>

    <form method="post" action="/ask">
        <input
            type="text"
            name="message"
            placeholder="How many samples are there for each cancer type?"
            value="{{ message|default('') }}"
            required
        />
        <button type="submit">Send</button>
    </form>

    {% if error %}
    <div class="box">
        <div class="error">Error</div>
        <pre>{{ error }}</pre>
    </div>
    {% endif %}

    {% if answer %}
    <div class="box">
        <h3>Answer</h3>
        <pre>{{ answer }}</pre>
    </div>
    {% endif %}

    {% if sql %}
    <div class="box">
        <h3>Generated SQL</h3>
        <pre>{{ sql }}</pre>
    </div>
    {% endif %}

    {% if result %}
    <div class="box">
        <h3>Query Result</h3>
        <pre>{{ result }}</pre>
    </div>
    {% endif %}
</body>
</html>
"""

# -----------------------------
# Database helpers
# -----------------------------
def get_connection():
    return oracledb.connect(
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=DB_DSN
    )

def extract_sql(text: str) -> str:
    """
    Extract SQL from Claude response.
    Supports plain SQL or fenced ```sql blocks.
    """
    if not text:
        return ""

    fence_match = re.search(r"```sql\s*(.*?)```", text, re.IGNORECASE | re.DOTALL)
    if fence_match:
        return fence_match.group(1).strip()

    generic_fence_match = re.search(r"```\s*(.*?)```", text, re.DOTALL)
    if generic_fence_match:
        return generic_fence_match.group(1).strip()

    return text.strip()

def validate_sql(sql: str) -> None:
    if not sql:
        raise ValueError("Claude returned an empty SQL query.")

    sql_clean = sql.strip().rstrip(";")
    sql_upper = sql_clean.upper()

    forbidden = [
        "INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE",
        "MERGE", "GRANT", "REVOKE", "CREATE", "RENAME", "BEGIN", "DECLARE"
    ]

    if not sql_upper.startswith("SELECT"):
        raise ValueError("Only SELECT statements are allowed.")

    for keyword in forbidden:
        if re.search(rf"\b{keyword}\b", sql_upper):
            raise ValueError(f"Forbidden SQL keyword detected: {keyword}")

def run_sql(sql: str, max_rows: int = 50) -> dict:
    validate_sql(sql)

    connection = get_connection()
    try:
        cursor = connection.cursor()
        cursor.execute(sql)
        columns = [col[0] for col in cursor.description] if cursor.description else []
        rows = cursor.fetchmany(max_rows)
        return {
            "columns": columns,
            "rows": rows,
            "row_count_returned": len(rows)
        }
    finally:
        connection.close()

# -----------------------------
# LLM helpers
# -----------------------------
def generate_sql(user_question: str) -> str:
    sql_prompt = f"""
Convert this question into a safe Oracle SQL query.

Question:
{user_question}

Return SQL only.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=400,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": sql_prompt}]
    )

    text = "".join(
        block.text for block in response.content if hasattr(block, "text")
    ).strip()

    sql = extract_sql(text)
    validate_sql(sql)
    return sql

def explain_result(user_question: str, sql_query: str, result: dict) -> str:
    explanation_prompt = f"""
User question:
{user_question}

SQL used:
{sql_query}

Query result:
{json.dumps(result, default=str, indent=2)}

Provide a concise explanation of the answer in plain English.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": explanation_prompt}]
    )

    answer = "".join(
        block.text for block in response.content if hasattr(block, "text")
    ).strip()

    return answer

def process_question(user_question: str) -> dict:
    if not user_question or not user_question.strip():
        raise ValueError("Question is empty.")

    sql_query = generate_sql(user_question)
    result = run_sql(sql_query)
    answer = explain_result(user_question, sql_query, result)

    return {
        "sql": sql_query,
        "result": result,
        "answer": answer
    }

# -----------------------------
# Routes
# -----------------------------
@app.route("/", methods=["GET"])
def home():
    return render_template_string(HTML_PAGE)

@app.route("/ask", methods=["POST"])
def ask():
    message = request.form.get("message", "").strip()

    try:
        output = process_question(message)
        return render_template_string(
            HTML_PAGE,
            message=message,
            answer=output["answer"],
            sql=output["sql"],
            result=json.dumps(output["result"], default=str, indent=2)
        )
    except Exception as e:
        return render_template_string(
            HTML_PAGE,
            message=message,
            error=str(e)
        ), 500

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.get_json(silent=True) or {}
        user_question = (data.get("message") or "").strip()
        output = process_question(user_question)
        return jsonify(output)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# -----------------------------
# App start
# -----------------------------
if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001, debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [17/Mar/2026 10:48:03] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 10:48:38] "POST /ask HTTP/1.1" 500 -
127.0.0.1 - - [17/Mar/2026 13:51:47] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 13:51:54] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 13:53:56] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 13:56:25] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 13:57:48] "POST /ask HTTP/1.1" 500 -
127.0.0.1 - - [17/Mar/2026 13:57:57] "POST /ask HTTP/1.1" 500 -
